# Autonomous Mower — YOLOv8n-seg Segmentation

This notebook trains a **YOLOv8n-seg** model on your Roboflow lawn dataset.

| | DeepLabV3-ResNet50 (old) | YOLOv8n-seg (this notebook) |
|---|---|---|
| Parameters | 40 M | 3.4 M |
| Model size | ~160 MB | ~7 MB |
| Jetson Nano FPS | 2-5 | 15-25 |
| TensorRT export | Manual | Built-in |

**Before running:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# Step 0: Install dependencies
!pip install -q ultralytics roboflow opencv-python-headless pyyaml

In [ ]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
# Step 1: Download dataset from Roboflow (png-mask-semantic format)
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("malhanaf-s-workspace").project("lawn-recognition-9e8gn")
version = project.version(1)
dataset = version.download("png-mask-semantic")

DATASET_DIR = dataset.location
print(f"\nDataset at: {DATASET_DIR}")

In [ ]:
# Step 2: Inspect the raw dataset
from pathlib import Path

train_dir = Path(DATASET_DIR) / "train"
classes_csv = train_dir / "_classes.csv"
print("Classes CSV:")
print(classes_csv.read_text())

for split in ["train", "valid", "test"]:
    imgs = [f for f in (Path(DATASET_DIR) / split).iterdir()
            if f.suffix == ".jpg" and "_mask" not in f.name]
    print(f"{split:6s}: {len(imgs)} images")

## Dataset Samples (for report)

Show what the raw images and their ground-truth semantic masks look like before training.

In [ ]:
# Step 3: Convert masks → YOLO polygon format
#
# Roboflow gives us pixel masks (image_mask.png). YOLOv8 needs polygon
# annotations in .txt files. This cell extracts contours from each mask
# and writes them as normalised YOLO polygon labels.

import csv
import shutil

import cv2
import numpy as np
import yaml


def mask_to_yolo_polygons(mask, img_h, img_w, min_area=100):
    results = []
    for pixel_val in np.unique(mask):
        if pixel_val == 0:
            continue
        binary = (mask == pixel_val).astype(np.uint8)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            if cv2.contourArea(contour) < min_area:
                continue
            epsilon = 0.005 * cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, epsilon, True)
            if len(approx) < 3:
                continue
            pts = approx.squeeze()
            coords = []
            for x, y in pts:
                coords.extend([round(x / img_w, 6), round(y / img_h, 6)])
            results.append((int(pixel_val) - 1, coords))  # 0-index for YOLO
    return results


def convert_split(src_dir, split, out_dir):
    src = Path(src_dir) / split
    if not src.exists():
        return 0
    img_out = out_dir / "images" / split
    lbl_out = out_dir / "labels" / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    files = {p.name: p for p in src.iterdir() if p.is_file()}
    count = 0
    for name, path in sorted(files.items()):
        if name.startswith("_") or "_mask" in name:
            continue
        if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        mask_name = f"{path.stem}_mask.png"
        if mask_name not in files:
            continue
        mask = cv2.imread(str(files[mask_name]), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        h, w = mask.shape
        polygons = mask_to_yolo_polygons(mask, h, w)
        shutil.copy2(path, img_out / path.name)
        with open(lbl_out / f"{path.stem}.txt", "w") as f:
            for cls_id, coords in polygons:
                f.write(f"{cls_id} " + " ".join(str(c) for c in coords) + "\n")
        count += 1
    return count


# Read class names (skip background at pixel 0)
class_names = {}
with open(classes_csv) as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        pv = int(row[0].strip())
        nm = row[1].strip()
        if pv > 0:
            class_names[pv - 1] = nm

YOLO_DIR = Path("lawn_yolo")
YOLO_DIR.mkdir(exist_ok=True)

for split in ["train", "valid", "test"]:
    n = convert_split(DATASET_DIR, split, YOLO_DIR)
    print(f"  {split}: {n} images converted")

DATA_YAML = YOLO_DIR / "data.yaml"
yaml_data = {
    "path": str(YOLO_DIR.resolve()),
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "names": class_names,
}
with open(DATA_YAML, "w") as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print(f"\ndata.yaml written to {DATA_YAML}")
print(f"Classes: {class_names}")

In [ ]:
# Quick sanity check: view a converted label
sample_lbl = sorted((YOLO_DIR / "labels" / "train").glob("*.txt"))[0]
print(f"Sample label ({sample_lbl.name}):")
print(sample_lbl.read_text()[:500])

In [ ]:
# Step 4: Train YOLOv8n-seg
import os
from pathlib import Path
from ultralytics import YOLO

os.chdir("/content")

model = YOLO("yolov8n-seg.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/mower_runs",
    name="lawn_seg",
    exist_ok=True,
    patience=10,
    save=True,
    plots=True,
)

RUN_DIR = Path(results.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"

print(f"\n{'='*50}")
print(f"  RUN_DIR  = {RUN_DIR}")
print(f"  best.pt  = {BEST_PT}")
print(f"  exists?  = {BEST_PT.exists()}")
print(f"{'='*50}")

In [ ]:
# Step 5: View training curves (auto-generated by Ultralytics)
from pathlib import Path
from IPython.display import Image, display

results_img = RUN_DIR / "results.png"
if results_img.exists():
    display(Image(filename=str(results_img), width=900))
else:
    print(f"results.png not at {results_img}")
    print("Contents of RUN_DIR:")
    for f in sorted(RUN_DIR.iterdir()):
        print(f"  {f.name}")

## Evaluation

- **Validation** = used during training to pick the best checkpoint
- **Test** = touched only once, right now, for the final unbiased score

Key metrics:
- **mAP50-seg**: mean Average Precision at IoU=0.50 for masks
- **mAP50-95-seg**: mean AP averaged over IoU 0.50→0.95 (stricter)

In [ ]:
# Step 6: Evaluate on VALIDATION set
from ultralytics import YOLO

best_model = YOLO(str(BEST_PT))

val_metrics = best_model.val(data=str(DATA_YAML), split="val", device=0)

print(f"\n{'='*40}")
print(f"  VALIDATION RESULTS")
print(f"{'='*40}")
print(f"  mAP50 (seg) : {val_metrics.seg.map50:.4f}")
print(f"  mAP50-95 (seg): {val_metrics.seg.map:.4f}")

In [ ]:
# Step 7: Evaluate on TEST set (final unbiased score — report this)
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    device=0,
)

print(f"\n{'='*40}")
print(f"  FINAL TEST SET RESULTS")
print(f"{'='*40}")
print(f"  mAP50 (seg) : {test_metrics.seg.map50:.4f}")
print(f"  mAP50-95 (seg): {test_metrics.seg.map:.4f}")
print(f"\n  Report these numbers in your capstone.")

In [ ]:
# Step 8: Visualize predictions on test images (report-ready)
import matplotlib.pyplot as plt
import cv2
import numpy as np

test_images = sorted(
    p for p in (YOLO_DIR / "images" / "test").iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)[:6]

fig, axes = plt.subplots(len(test_images), 2, figsize=(14, 5 * len(test_images)))
if len(test_images) == 1:
    axes = [axes]

CLASS_COLORS = {0: (255, 0, 0), 1: (0, 0, 255), 2: (0, 200, 0)}
CLASS_NAMES = {0: "barriers", 1: "boundary", 2: "lawn"}

for i, img_path in enumerate(test_images):
    preds = best_model.predict(source=str(img_path), conf=0.25, device=0, verbose=False)
    result = preds[0]
    orig = cv2.cvtColor(result.orig_img, cv2.COLOR_BGR2RGB)
    h, w = orig.shape[:2]

    overlay = orig.copy()
    if result.masks is not None:
        for j, mask in enumerate(result.masks.data):
            cls_id = int(result.boxes.cls[j])
            color = CLASS_COLORS.get(cls_id, (255, 255, 0))
            binary = cv2.resize(mask.cpu().numpy(), (w, h))
            overlay[binary > 0.5] = (
                overlay[binary > 0.5] * 0.4 + np.array(color) * 0.6
            ).astype(np.uint8)

    axes[i][0].imshow(orig)
    axes[i][0].set_title(f"Input: {img_path.name}", fontsize=12)
    axes[i][0].axis("off")
    axes[i][1].imshow(overlay)
    axes[i][1].set_title("Segmentation", fontsize=12)
    axes[i][1].axis("off")

legend_patches = [
    plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=np.array(c)/255,
               markersize=12, label=n)
    for c, n in zip(CLASS_COLORS.values(), CLASS_NAMES.values())
]
fig.legend(handles=legend_patches, loc="lower center", ncol=3, fontsize=12,
           bbox_to_anchor=(0.5, -0.02))
plt.suptitle("YOLOv8n-seg — Test Set Segmentation Results", fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig("segmentation_results.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved to segmentation_results.png")

In [ ]:
# Step 9: Export to ONNX for Jetson Nano deployment
onnx_path = best_model.export(
    format="onnx",
    imgsz=640,
    simplify=True,
)
print(f"ONNX model saved to: {onnx_path}")
print(f"\nOn Jetson Nano, convert ONNX → TensorRT for maximum speed:")
print(f"  /usr/src/tensorrt/bin/trtexec --onnx=best.onnx --saveEngine=best.engine --fp16")

In [ ]:
# Step 10: Download results
from google.colab import files
import shutil

shutil.make_archive("mower_seg_results", "zip", str(RUN_DIR))

files.download(str(BEST_PT))
files.download(str(onnx_path))
files.download("segmentation_results.png")
files.download("mower_seg_results.zip")
if (RUN_DIR / "results.png").exists():
    files.download(str(RUN_DIR / "results.png"))

print("\nDownloaded files:")
print("  best.pt                   → YOLOv8n-seg weights")
print("  best.onnx                 → ONNX export for Jetson")
print("  segmentation_results.png  → visual results for report")
print("  mower_seg_results.zip     → full training run (curves, metrics, weights)")